In [35]:
import pandas as pd
import requests
from io import BytesIO
import numpy as np
import os
import re
import openpyxl

In [36]:
realtor = pd.read_csv("https://econdata.s3-us-west-2.amazonaws.com/Reports/Core/RDC_Inventory_Core_Metrics_County_History.csv")
realtor_hotness = pd.read_csv("https://econdata.s3-us-west-2.amazonaws.com/Reports/Hotness/RDC_Inventory_Hotness_Metrics_County_History.csv")

url = "https://www.freddiemac.com/pmms/docs/historicalweeklydata.xlsx"
headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"}
response = requests.get(url, headers=headers)
response.raise_for_status()
mortgage_rates = pd.read_excel(BytesIO(response.content))

In [37]:
common_cols = set(realtor.columns) & set(realtor_hotness.columns)
merge_keys = {"month_date_yyyymm", "county_fips"}
cols_to_drop = common_cols - merge_keys
realtor_hotness = realtor_hotness.drop(columns=list(cols_to_drop))

In [ ]:
mortgage_rates_df_clean = mortgage_rates.copy(deep=True)
mortgage_rates_df_clean = mortgage_rates_df_clean[mortgage_rates_df_clean['Unnamed: 0'].notnull()]

# take first row as header
mortgage_rates_df_clean.columns = mortgage_rates_df_clean.iloc[0].tolist()
mortgage_rates_df_clean = mortgage_rates_df_clean[1:].reset_index(drop=True)

# pull weeks from timestamps
week_parsed = pd.to_datetime(mortgage_rates_df_clean["Week"], errors="coerce")
mortgage_rates_df_clean = mortgage_rates_df_clean[week_parsed.notna()].copy()
mortgage_rates_df_clean["Week"] = week_parsed[week_parsed.notna()].dt.date

# renaming columns from base spreadsheet
mortgage_rates_df_clean.columns.values[0] = "Week"
mortgage_rates_df_clean.columns.values[1] = "U.S. 30 year FRM"

# truncate to recent dates
mortgage_rates_df_clean = mortgage_rates_df_clean[mortgage_rates_df_clean['Week']>=pd.to_datetime("2016-01-01").date()]

# move from week to month level
mortgage_rates_df_clean['Week'] = pd.to_datetime(mortgage_rates_df_clean['Week'])
mortgage_rates_df_clean['Month'] = mortgage_rates_df_clean['Week'].dt.to_period('M').dt.to_timestamp()
mortgage_rates_df_clean.drop(columns=['Week'], inplace=True)
mortgage_rates_df_clean = mortgage_rates_df_clean.groupby('Month').mean().reset_index()

In [41]:
realtor_merge = pd.merge(realtor, realtor_hotness, how="outer", on=["month_date_yyyymm", "county_fips"])
realtor_merge[['city', 'state']] = realtor_merge['county_name'].str.split(', ', expand=True)
realtor_merge['month_date_yyyymm'] = pd.to_datetime(realtor_merge['month_date_yyyymm'], format='%Y%m')

In [42]:
processed_data_pre_model = pd.merge(realtor_merge, mortgage_rates_df_clean, left_on="month_date_yyyymm", right_on="Month", how="left")

In [43]:
processed_data_pre_model.rename(columns={"month_date_yyyymm": "date"}, inplace=True)

In [44]:
processed_data_pre_model.to_csv("./data/processed/processed_data_pre_model.csv", index=False)